# 07 Advanced & Agentic RAG Paradigms

**Real-World Scenario**: Web-Augmented Financial Research Agent.

This notebook demonstrates a **Corrective RAG (CRAG) Decision Router** (evaluating internal vector search confidence; falling back to Tavily Web Search API if internal context relevance is low), completed by a **Complete GenAI Financial Research Agent Call**.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Corrective RAG (CRAG) Router State Machine Code

In [ ]:
def crag_decision_router(query: str, retrieved_docs: list[str], confidence_threshold: float = 0.70) -> str:
    has_keywords = any("Nvidia" in d or "revenue" in d for d in retrieved_docs)
    confidence = 0.90 if has_keywords else 0.35
    
    if confidence >= confidence_threshold:
        return "GENERATE_ANSWER"
    else:
        return "WEB_SEARCH_FALLBACK"

query = "What is Nvidia Q4 revenue forecast?"
retrieved = ["Intel CPUs released in 2025.", "AMD market share growth in server market."]
action = crag_decision_router(query, retrieved)

print(f"User Query: '{query}'")
print(f"Retrieved Chunks Confidence: LOW (<0.70)")
print("=== Corrective RAG (CRAG) Action Triggered ===")
print(f"Triggered Action: {action} (Internal context irrelevant -> Invoking Tavily Web Search)")


## Step 3: Executing Complete GenAI Financial Research Agent Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if os.getenv("OPENAI_API_KEY"):
    prompt = ChatPromptTemplate.from_template("You are an Agentic Financial Analyst. State that internal context was insufficient and action taken: {action} for Query: {query}.")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    response = llm.invoke(prompt.format(action=action, query=query))
    print("\n=== Complete CRAG Agent GenAI Response ===")
    print(response.content)
